In [1]:
from langgraph.graph import END, START, StateGraph
from langgraph.constants import Send

In [2]:
import operator
from typing import Annotated, Dict, List

from pydantic import BaseModel, Field


class MainGraphState(BaseModel):
    criterion: Annotated[List[str], operator.add]
    title: str
    criteria: List[Dict[str, str]] = Field(default_factory=lambda: [
        {"id": "1", "description": "Criterion 1", "context": "Context 1"},
        {"id": "2", "description": "Criterion 2", "context": "Context 2"},
        {"id": "3", "description": "Criterion 3", "context": "Context 3"},
    ])

    model_config = {
        "arbitrary_types_allowed": True
    }

class SubGraphState(BaseModel):
    main_state: MainGraphState
    criterion_description: str
    criterion_context: str
    criterion_id: str


In [3]:
def node_1(state: SubGraphState) -> SubGraphState:
    return state

def node_2(state: SubGraphState) -> MainGraphState:
    return {"criterion": [state.criterion_description]}

def get_subgraph():
    # Define a new graph
    workflow = StateGraph(
        SubGraphState, input=SubGraphState, output=MainGraphState
    )

    # Define the nodes
    workflow.add_node("node_1", node_1)
    workflow.add_node("node_2", node_2)

    # Set the entrypoint as `init_agent`
    workflow.add_edge(START, "node_1")
    workflow.add_edge("node_1", "node_2")
    workflow.add_edge("node_2", END)

    # Compile the graph
    graph = workflow.compile()
    return graph


In [4]:
def continue_to_subgraph(state: MainGraphState):
    print(state)
    return [
        Send(
            "subgraph",
            {
                "main_state": state,
                "criterion_description": c["description"],
                "criterion_context": c["context"],
                "criterion_id": c["id"],
            },
        )
        for c in state.criteria
    ]

In [5]:
def basic_node(state: MainGraphState):
    return state

def get_main_graph():
    workflow = StateGraph(
        MainGraphState, input=MainGraphState, output=MainGraphState
    )

    workflow.add_node("basic_node", basic_node)
    workflow.add_node("subgraph", get_subgraph())
    
    workflow.add_edge(START, "basic_node")
    workflow.add_conditional_edges(
        "basic_node", continue_to_subgraph, ["subgraph"]
    )
    workflow.add_edge("subgraph", END)

    return workflow.compile()

/Users/aberman/Documents/Workshop/repio-intelligence/.venv/lib/python3.12/site-packages/pydantic/_internal/_generate_schema.py:547: UserWarning: <built-in function any> is not a Python type (it may be an instance of an object), Pydantic will allow any object with no validation since we cannot even enforce that the input is an instance of the given type. To get rid of this error wrap the type with `pydantic.SkipValidation`.
  warn(


In [6]:
graph = get_main_graph()
graph.invoke({"title": "test"})

criterion=[] title='test' criteria=[{'id': '1', 'description': 'Criterion 1', 'context': 'Context 1'}, {'id': '2', 'description': 'Criterion 2', 'context': 'Context 2'}, {'id': '3', 'description': 'Criterion 3', 'context': 'Context 3'}]
Context 2
Context 1
Context 3


GraphRecursionError: Recursion limit of 25 reached without hitting a stop condition. You can increase the limit by setting the `recursion_limit` config key.